# Image Classification in 2D Images: Computer Vision 101, Going from Zero to Hero

Welcome to the foundation of modern Computer Vision. In this tutorial, we will build a Convolutional Neural Network (CNN) to classify images into 10 distinct categories. 

We will use the **CIFAR-10** dataset, a gold standard in machine learning consisting of 60,000 32x32 color images (e.g., airplanes, cars, birds, cats). 

### Our Analytical Pipeline:
1. **Data Ingestion:** Load and visualize the pixel data.
2. **Preprocessing:** Normalize the tensors for mathematical stability.
3. **Architecture Design:** Build a CNN using spatial hierarchies (Convolutions and Pooling).
4. **Compilation & Training:** Apply Gradient Descent to optimize our filter weights.
5. **Evaluation:** Analyze the model's accuracy and visualize predictions.

In [ ]:
# Cell 2: Environment Setup and Imports
# We use TensorFlow and its high-level API, Keras, to build our neural network.
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np

# Verify TensorFlow is working
print(f"TensorFlow Version: {tf.__version__}")

## Step 1: Data Ingestion and Exploration

A neural network is only as good as the data fed into it. The CIFAR-10 dataset comes pre-split into 50,000 training images and 10,000 testing images. 

Let's load the data and visually inspect a few samples to understand the complexity of the task. These images are very low resolution (32x32 pixels), which makes classification challenging even for humans!

In [ ]:
# Cell 4: Loading CIFAR-10

# Load the dataset directly from Keras datasets
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.cifar10.load_data()

# Define the human-readable class names (CIFAR-10 labels are just integers 0-9)
class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

print(f"Training Data Shape: {train_images.shape}") # Output: (50000, 32, 32, 3) -> 50k images, 32x32 pixels, 3 color channels (RGB)

# Plot the first 5 images in the training set
plt.figure(figsize=(15, 3))
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.xticks([]) # Hide axes for cleaner look
    plt.yticks([])
    plt.imshow(train_images[i])
    # Extract the scalar label from the 2D array and map to class name
    plt.xlabel(class_names[train_labels[i][0]], fontsize=12)
plt.show()

## Step 2: Data Preprocessing

Image pixels are represented by numbers ranging from 0 to 255 (where 0 is black, and 255 is maximum color intensity). 

If we feed these raw, large numbers into a neural network, the gradient descent algorithms will suffer from severe mathematical instability (exploding gradients) and take a very long time to converge. 

**Analytical Step:** We must **Normalize** the data to a scale between 0.0 and 1.0. We do this simply by dividing every pixel value by 255.0.

In [ ]:
# Cell 6: Normalization

# Convert the integer pixel values to 32-bit floats and scale them
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0

print(f"Pixel value range after normalization: {np.min(train_images)} to {np.max(train_images)}")

## Step 3: Designing the CNN Architecture

This is the core of Computer Vision. We will build a sequential model with two main segments:

1. **The Feature Extractor (Convolutional Base):**
   * **Conv2D Layers:** These apply the sliding filters (as seen in the interactive widget above) to learn spatial features like edges, curves, and eventually complex shapes like ears or wheels.
   * **MaxPooling2D Layers:** These drastically reduce the spatial dimensions (width and height) of the image maps, reducing computational load and forcing the network to learn translation-invariant features (e.g., recognizing a cat regardless of where it is in the frame).

2. **The Classifier (Dense Layers):**
   * **Flatten:** Converts the 2D feature maps into a flat 1D vector.
   * **Dense:** Standard fully connected layers that take the extracted features and output probabilities for our 10 classes.

In [ ]:
# Cell 8: Building the Model

model = models.Sequential()

# --- FEATURE EXTRACTOR ---
# 1st Convolutional Block: 32 filters, 3x3 kernel, ReLU activation
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
model.add(layers.MaxPooling2D((2, 2))) # Shrinks feature map by half

# 2nd Convolutional Block: 64 filters
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

# 3rd Convolutional Block: 64 filters
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

# --- CLASSIFIER ---
model.add(layers.Flatten()) # Unroll the 3D tensor into 1D
model.add(layers.Dense(64, activation='relu')) # Hidden layer for deep reasoning
model.add(layers.Dense(10)) # Output layer: 10 neurons for 10 classes

# Display the architecture analytically
model.summary()

## Step 4: Compiling and Training

Before the model can learn, it needs three mathematical definitions:
1. **Optimizer (`adam`):** The algorithm that updates the network weights based on the data it sees and its loss function.
2. **Loss Function (`SparseCategoricalCrossentropy`):** Measures how accurate the model is during training. We want to minimize this value. (We use *Sparse* because our labels are integers, not one-hot encoded vectors).
3. **Metrics (`accuracy`):** To monitor the training and testing steps.

We will train the model for 10 epochs (complete passes through the 50,000 images).

In [ ]:
# Cell 10: Compile and Fit

model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

print("Starting Model Training...")

# We pass the test set as validation data to monitor overfitting during training
history = model.fit(train_images, train_labels, 
                    epochs=10, 
                    validation_data=(test_images, test_labels))

print("Training Complete!")

## Step 5: Evaluation and Analytical Conclusion

Training accuracy tells us how well the model memorized the training data, but **Validation Accuracy** tells us how well it generalizes to *unseen* data.

If Training Accuracy is very high (e.g., 95%) but Validation Accuracy is low (e.g., 60%), the model is **overfitting** (memorizing noise rather than learning general patterns). 

Let's plot the learning curves to analyze our model's health and visualize a real prediction.

In [ ]:
# Cell 12: Evaluation Visualizations

# 1. Plotting the Learning Curve
plt.figure(figsize=(10, 4))
plt.plot(history.history['accuracy'], label='Training Accuracy', color='blue')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='orange')
plt.title('Model Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

# 2. Making a Prediction on an unseen image
image_index = 42 # Pick a random test image
sample_image = test_images[image_index]
true_label = class_names[test_labels[image_index][0]]

# The model expects a batch, so we expand dimensions from (32,32,3) to (1,32,32,3)
prediction_logits = model.predict(np.expand_dims(sample_image, axis=0))

# Apply Softmax to convert raw logits into percentages/probabilities
probabilities = tf.nn.softmax(prediction_logits[0])
predicted_class = class_names[np.argmax(probabilities)]

# Visualize the result
plt.figure(figsize=(4, 4))
plt.imshow(sample_image)
plt.title(f"True: {true_label} | Predicted: {predicted_class}")
plt.axis('off')
plt.show()

print(f"Confidence score: {np.max(probabilities) * 100:.2f}%")